# XP Phase 2 spike (throwaway): MiniGrid DoorKey timing and size floor
Runs PPO (Kairi-style) and PPO+RND (Kaiser-style) on DoorKey-6x6, then evaluates zero-shot at sizes 5/6/8/12 against a random policy.
Answers: how long does a run take, and are sizes 8 and 12 stuck at 0% success?

Use a CPU runtime (the env is CPU-bound). Results are saved to Drive so a disconnect loses nothing: rerun and finished runs are skipped.
Does not install the project's pinned numpy, so no runtime restart should be needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/xp-spike

In [ ]:
!rm -rf xp-agents && git clone --depth 1 https://github.com/knguyen0923/xp-agents xp-agents
%cd xp-agents
!pip install -q minigrid

In [ ]:
# 500k steps per run (use --steps 100000 for a quick timing-only pass). Rerun-safe: skips if done.
import os
%cd /content/xp-agents
if not os.path.exists('/content/drive/MyDrive/xp-spike/kairi_s0.json'):
    !python spikes/spike_minigrid.py --train-size 6 --steps 500000 --seed 0 --out /content/drive/MyDrive/xp-spike/kairi_s0.json
else:
    print('kairi run already on Drive, skipping')

In [ ]:
# 500k steps per run (use --steps 100000 for a quick timing-only pass). Rerun-safe: skips if done.
import os
%cd /content/xp-agents
if not os.path.exists('/content/drive/MyDrive/xp-spike/kaiser_s0.json'):
    !python spikes/spike_minigrid.py --train-size 6 --steps 500000 --seed 0 --rnd --out /content/drive/MyDrive/xp-spike/kaiser_s0.json
else:
    print('kaiser run already on Drive, skipping')

In [ ]:
OUT = '/content/drive/MyDrive/xp-spike'
SEED = 0
import json, pandas as pd
rows = []
for name in ('kairi', 'kaiser'):
    r = json.load(open(f'{OUT}/{name}_s{SEED}.json'))
    t = r['train']
    rows.append({'agent': name, 'env_only_sps': round(r['env_only_sps']), 'train_sps': round(t['sps']), 'train_minutes': round(t['seconds'] / 60, 1), 'steps_to_80pct': t['steps_to_80pct_success'],
                 **{f'size{s}_trained': r['eval_zero_shot'][s]['success'] for s in r['eval_zero_shot']},
                 **{f'size{s}_random': r['random_baseline'][s]['success'] for s in r['random_baseline']}})
pd.set_option('display.width', 200)
summary = pd.DataFrame(rows)
print(summary.T)
print(r['versions'])

## RND diagnostic v2 (250k steps each, about 6 min)
Rerun the clone/install cell first so the repo has the new `ep_len` and `bonus/step` logging (a stale clone omits them). v1 (150k steps) was inconclusive: even coef 0.0 had not learned by 150k, so those runs could not separate the bonus effect from slow sparse-reward learning. v2 adds a plain-PPO control with the same seed and runs longer.
Read: `success100`, `ep_len` (cap 360), `bonus/step`.

In [ ]:
import os
%cd /content/xp-agents
if not os.path.exists('/content/drive/MyDrive/xp-spike/diag2_control.json'):
    !python spikes/spike_minigrid.py --train-size 6 --steps 250000 --seed 0 --eval-sizes 6 --out /content/drive/MyDrive/xp-spike/diag2_control.json
else:
    print('control already on Drive, skipping')

In [ ]:
import os
%cd /content/xp-agents
if not os.path.exists('/content/drive/MyDrive/xp-spike/diag2_coef0p1.json'):
    !python spikes/spike_minigrid.py --train-size 6 --steps 250000 --seed 0 --rnd --rnd-coef 0.1 --eval-sizes 6 --out /content/drive/MyDrive/xp-spike/diag2_coef0p1.json
else:
    print('coef0p1 already on Drive, skipping')

In [ ]:
import os
%cd /content/xp-agents
if not os.path.exists('/content/drive/MyDrive/xp-spike/diag2_coef0p01.json'):
    !python spikes/spike_minigrid.py --train-size 6 --steps 250000 --seed 0 --rnd --rnd-coef 0.01 --eval-sizes 6 --out /content/drive/MyDrive/xp-spike/diag2_coef0p01.json
else:
    print('coef0p01 already on Drive, skipping')